# 第62章 聚类热力图（clustermap）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 19 / 20 步：组织多变量、矩阵和分面证据**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 热力图（heatmap）  →  **本章任务：** 聚类热力图（clustermap）  →  **下一步：** 分面图（FacetGrid）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

当一份表格既有很多行（比如不同品类）也有很多列（比如不同区域）时，逐格看数字很难发现规律。


## 本章目标

学完本章，你将能够：

- **理解**：理解「聚类热力图（clustermap）」的核心思想、适用场景、关键假设与要解释的业务问题。
- **操作**：能按标准流程完成数据准备、模型训练与评估，并解读「聚类热力图（clustermap）」的关键输出指标。
- **迁移**：能把「聚类热力图（clustermap）」迁移到一份新数据上，独立完成任务并就结果给出有分寸的结论。


## 适用场景

**背景引入**：当一份表格既有很多行（比如不同品类）也有很多列（比如不同区域）时，逐格看数字很难发现规律。聚类热力图会先计算相似度，把表现相近的行和列自动归并到一起，再用颜色深浅把数值高低在一张图里铺开，让隐藏在矩阵里的群组和整体模式一目了然。它很适合作为探索的第一步——在深入分析之前，先看清「谁和谁像」。

行列数量较多，希望按相似模式自动分组。


## 数据结构

行和列均为可比较的数值矩阵；通常需要标准化。


## 本章练习任务

运行基础图表后，完成以下任务：

1. 将 z_score=1 改为 standard_scale=0，对比按列标准化与按行标准化的聚类结果
2. 修改 row_cluster=True 为 row_cluster=False，观察禁用行聚类对树状图的影响
3. 调整 method 参数（如 'average' 或 'complete'），说明不同连接方法对聚类结构的影响


## 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `orders.pivot_table()`、`sns.clustermap()`、`grid.fig.suptitle()`、`plt.show()` | 行列数量较多，希望按相似模式自动分组。 | 量纲不同却不标准化 |
| 进阶变体 | `marketing.groupby()`、`sns.clustermap()`、`grid.fig.suptitle()`、`plt.show()` | 在基础图表上增加分组、注释、布局或交互 | 把聚类结果当作唯一真实分类 |
| 关键参数 | `z_score/standard_scale` | 标准化 | 量纲不同却不标准化 |
| 关键参数 | `method` | 连接方法 | 把聚类结果当作唯一真实分类 |
| 关键参数 | `metric` | 距离 | 样本太少或缺失太多 |
| 关键参数 | `row_cluster/col_cluster` | 聚类方向 | 量纲不同却不标准化 |


## 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-62 -->
### 数学推导｜距离决定聚类结果

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜先计算逐特征差异。** $\Delta_j=x_j-y_j$。

**第 2 步｜选择如何汇总差异。** 欧氏距离对大差异平方后更敏感，曼哈顿距离把绝对差异直接相加。

**第 3 步｜看尺度为什么重要。** 若改用标准化坐标 $z_j=(x_j-\mu_j)/\sigma_j$，则欧氏距离变为

$$
d_z(x,y)=\sqrt{\sum_j\left(\frac{x_j-y_j}{\sigma_j}\right)^2}
$$

这等于让每一维按自身尺度参与比较，避免大单位变量天然主导距离。

**把上面的关系收束为本章计算式：**

$$
d_2(x,y)=\sqrt{\sum_j(x_j-y_j)^2},\qquad d_1(x,y)=\sum_j|x_j-y_j|
$$

**符号解释：** $d_2$ 是欧氏距离，$d_1$ 是曼哈顿距离。

**代码对应：** 聚类前标准化特征，并说明距离与 linkage/邻域参数的选择。

**使用边界：** 不同量纲会让大数值特征主导距离；聚类簇不是天然存在的真实类别。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(
    f'Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行'
)


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(),
        "高于中位价",
        "不高于中位价",
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(
    f'样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行'
)


## 例 1｜最小可用图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import scipy

import matplotlib.pyplot as plt

category_region = orders.pivot_table(
    index="category", columns="region", values="order_value", aggfunc="mean"
).dropna()
grid = sns.clustermap(
    category_region,
    cmap="Blues",
    annot=True,
    fmt=".0f",
    figsize=(7, 6),
    row_cluster=True,
    col_cluster=True,
)
grid.fig.suptitle("品类与区域客单价聚类", y=1.02)
plt.show()


In [ ]:
# （自动维护）练习上下文快照 1：参考答案将基于此刻的变量运行
_pds_snap_1 = dict(globals())


**练一练**：刚才的基础图表按 `order_value`（客单价）对 category×region 做了聚类。现在把透视表的值字段从 `order_value` 换成 `items`（钻石克拉数），重画一张聚类热力图，观察换了一个数据字段后，行列的分组和颜色分布有什么变化。只改这一个字段，其余参数保持原样即可。


In [ ]:
try:
    pass
    # 请在下方填写代码：把透视表的 values 参数从 "order_value" 改为 "items"，并用 .dropna() 去掉存在空组合的行，其余保持不变。
    # TODO：请在下方完成 —— 练一练：刚才的基础图表按 order_value（客单价）对 category×region
    # 做了聚类。现在把透视表的值

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 例 2｜进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

profile = marketing.groupby("channel")[
    ["visits", "ad_spend", "sales", "conversion"]
].mean()
grid = sns.clustermap(
    profile,
    z_score=1,
    cmap="vlag",
    center=0,
    annot=True,
    fmt=".2f",
    figsize=(8, 6),
)
grid.fig.suptitle("渠道指标标准化聚类", y=1.02)
plt.show()


## 参数说明

- z_score/standard_scale：标准化
- method：连接方法
- metric：距离
- row_cluster/col_cluster：聚类方向


## 结果解读

树状图表达合并顺序和距离，色块表达标准化后的相对模式。


## 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 易错点提醒

- 量纲不同却不标准化
- 把聚类结果当作唯一真实分类
- 样本太少或缺失太多


## 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# （自动维护）练习上下文快照 2：参考答案将基于此刻的变量运行
_pds_snap_2 = dict(globals())


In [ ]:
try:
    # 独立迁移练习：换一种聚类距离，观察分组结果的变化
    # 【目标】聚类方法(method)不同会导致分组不同，练习体会它对结果的影响。
    import matplotlib.pyplot as plt
    import seaborn as sns

    # 起点示例(已可运行)：用 method 换成 complete 聚类。
    category_region = orders.pivot_table(
        index="category",
        columns="region",
        values="order_value",
        aggfunc="mean",
    ).dropna()
    grid = sns.clustermap(
        category_region,
        cmap="Blues",
        method="complete",
        figsize=(7, 6),
        row_cluster=True,
        col_cluster=True,
    )
    grid.fig.suptitle("品类与区域客单价聚类（complete）", y=1.02)
    plt.show()

    # ---- 反思记录：换成 complete 后聚类树的分支如何变化 ----
    change_note = "待填写"
    expected_change = "待填写"
    observed_change = "运行后填写"
    print(f"改动：{change_note}")
    print(f"预期：{expected_change}")
    print(f"观察：{observed_change}")

except Exception as _pds_err:
    print("（练习尚未完成或未填全：", _pds_err, "）")


## 小结

用层次聚类重新排列矩阵，发现相似行列和潜在群组。


### 你已经掌握

- 判断聚类热力图（clustermap）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 关键参数

| 参数 | 作用 |
| --- | --- |
| `z_score/standard_scale` | 标准化 |
| `method` | 连接方法 |
| `metric` | 距离 |
| `row_cluster/col_cluster` | 聚类方向 |


### 需要注意

- 量纲不同却不标准化
- 把聚类结果当作唯一真实分类
- 样本太少或缺失太多


## 参考答案


### 本章练习


In [ ]:
# 恢复练习 1 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_1)


### 本章练习


In [ ]:
# 恢复练习 2 时的变量上下文（后面的示例覆盖过这些名字）
globals().update(_pds_snap_2)


In [ ]:
import matplotlib.pyplot as plt

region_profile = orders.groupby("region").agg(
    order_value=("order_value", "mean"), items=("items", "mean")
)
grid = sns.clustermap(
    region_profile,
    standard_scale=1,
    cmap="YlGnBu",
    annot=True,
    fmt=".2f",
    figsize=(7, 5),
)
grid.fig.suptitle("区域订单特征聚类", y=1.02)
plt.show()
